[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-11.ipynb)

# 11주차 실습: 제공된 단계 점수로 Best-of-N 관찰

- 환경: Python 3, CPU, 표준 라이브러리, 인터넷 연결
- 모든 셀을 수정 없이 실행. 수업계획서·이슈 #4/#50에 따른 관찰 전용 활동
- GPU·모델 다운로드·추가 패키지·학생 작성 verifier 없음
- **후보 문장과 점수는 교수 작성 교육 예시**, 실제 ORM·PRM 출력 아님
- 학습·생성·성능 실험 없음. 계산 결과는 제공값에 대한 집계 관찰

## 출처

- [Let's Verify Step by Step](https://arxiv.org/html/2305.20050v1), 2.1·2.4·Appendix F.2
- [Math-Shepherd](https://arxiv.org/html/2312.08935v3), 자동 과정 감독
- [PRIME](https://arxiv.org/html/2502.01456v2), outcome 라벨 기반 암묵 PRM
- [위키백과 제주특별자치도 고정판 42464087](https://ko.wikipedia.org/w/index.php?oldid=42464087)
- 원문·저자 목록·이용 조건은 해당 문서와 문서 이력에서 확인. 본 실습은 원문을 runtime에 수집해 출처와 일부 근거 출력, 원문 전체 저장 없음


## 학습 기록 조건

- 학번: 202221015. GitHub 과제 제출용 관찰 기록.
- 관찰형 주차로 제공된 계산과 실험 조건 유지, 학생 알고리즘이나 verifier 추가 없음.
- 교수 작성 예시와 실제 공개 원문 확인, 실제 모델 생성 여부를 구분하여 기록.
- 한국어 위키백과 기여자들의 [제주특별자치도 고정판 42464087](https://ko.wikipedia.org/w/index.php?oldid=42464087), [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). 원문 전체 파일 저장 없음.
- 생성형 AI의 코드 실행 및 결과 정리 보조 사용. 학습자 개인의 이해도를 평가한 기록으로 주장하지 않음.
- 원본 코드의 연락처가 포함된 User-Agent만 공개 강의 저장소 URL로 변경.
- 재현 보완: Python 실행 조건과 출처 라이선스 메타데이터만 추가. 모델 호출과 파라미터 업데이트 없음.


## 1. 제주 공개 문서의 근거 확인

- 고정판 제목과 revision 확인 후 제주시·서귀포시·행정시를 함께 포함하는 실제 원문 문단 확인
- 고정판의 정적 행정정보를 이용한 교육 활동, 최신 운영정보 판정 도구 아님
- 네트워크 오류는 짧은 backoff 후 명시적 오류로 종료. 가짜 문서 fallback 없음
- 같은 출처·질문·후보·점수를 모든 집계 조건에서 유지


In [1]:
import hashlib
import html
import json
import math
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean

OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course-student/1.0 (https://github.com/halla-ai/deepnlp-2026)"})
for attempt in range(3):
    try:
        with urllib.request.urlopen(request, timeout=40) as response:
            raw_response = response.read()
        parsed = json.loads(raw_response)["parse"]
        break
    except (urllib.error.URLError, TimeoutError) as error:
        if attempt == 2:
            raise RuntimeError("공개 원문 수집 실패. 연결·출처 확인 후 다시 실행. 대체 문서 없음") from error
        time.sleep(2 ** attempt)
assert parsed["revid"] == OLDID, "고정판 번호 불일치"
assert parsed["title"] == "제주특별자치도", "원문 제목 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    fragment = re.sub(r"<[^>]+>", " ", fragment)
    return re.sub(r"\s+", " ", html.unescape(fragment)).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
evidence = [p for p in paragraphs if all(term in p for term in ("행정시", "제주시", "서귀포시"))]
assert evidence, "두 행정시의 원문 문단을 확인하지 못함. 자동 대체 금지"
source_metadata = {
    "url": SOURCE, "title": parsed["title"], "revision": parsed["revid"],
    "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    "response_sha256": hashlib.sha256(raw_response).hexdigest(),
    "attribution": "한국어 위키백과 제주특별자치도 문서와 해당 판의 기여자; 저자·이용 조건은 문서 이력 참조",
}
print("출처 메타데이터:", source_metadata)
print("실제 공개 원문 근거:", evidence[0])
EXPECTED = {"제주시", "서귀포시"}  # 위 원문과 대조할 공개 사실, 점수에서 도출하지 않음

import platform
execution_conditions = {"python": platform.python_version(), "machine": platform.machine(), "runtime": "CPU / Python standard library", "algorithm_changes": False}
source_metadata["license"] = "CC BY-SA 4.0"
source_metadata["history_url"] = "https://ko.wikipedia.org/w/index.php?title=제주특별자치도&action=history"
print("실행 조건:", execution_conditions)


출처 메타데이터: {'url': 'https://ko.wikipedia.org/w/index.php?oldid=42464087', 'title': '제주특별자치도', 'revision': 42464087, 'fetched_at_utc': '2026-11-12T15:02:45.085287+00:00', 'response_sha256': '8dd0d492441ff5c8b930eef682110b6736c472a119b0c935f3066462ac15f87f', 'attribution': '한국어 위키백과 제주특별자치도 문서와 해당 판의 기여자; 저자·이용 조건은 문서 이력 참조'}
실제 공개 원문 근거: 조선 시대에는 전라도 에 속했으나, 1946년 8월 전라남도 ,에서 분리되어 제주도(濟州道)로 승격되었다. 2006년 7월 1일부터 특별자치도 로 승격되어 자치경찰의 실시, 교육자치권의 확대, 일부 중앙권한의 이양, 자치입법권, 자치재정권의 부여 등 지역적 자치권이 고도화되었다. 하위에 기초지방자치단체 를 두지 않은 단층제 광역자치단체이며, 남북으로 두 개의 행정시 인 제주시 와 서귀포시 가 설치되어 있다.
실행 조건: {'python': '3.10.0', 'machine': 'AMD64', 'runtime': 'CPU / Python standard library', 'algorithm_changes': False}


## 2. 교수 작성 후보와 정답 대조를 분리

- 모든 후보는 같은 공개 정보 질문에 대한 교육용 문장
- 응답 점수와 단계 점수는 교수 제공값, 교정된 확률이나 신경망 출력 아님
- `answer_match`는 최종 답의 두 도시 집합을 공개 근거와 비교한 제한적 판정
- `process_review`는 교수의 단계 검토 설명, 자동 인과 검증 결과 아님
- A: 틀린 추론·맞는 최종 답. C/D: 앞 두 단계 타당·틀린 최종 답


In [2]:
candidates = [
    {"id": "A", "steps": ["문서에서 제주시와 서귀포시 명칭 확인", "이름이 다른 지역은 모두 행정시라고 간주", "제주시와 서귀포시라고 답변"],
     "answer": ["제주시", "서귀포시"], "outcome_score": 0.95, "step_scores": [0.9, 0.1, 0.9],
     "process_review": "2단계 범주 추론 오류. 최종 답 일치가 이 추론을 정당화하지 않음"},
    {"id": "B", "steps": ["공개 문서의 행정시 범주 확인", "제주시와 서귀포시 두 명칭을 해당 범주와 연결", "제주시와 서귀포시라고 답변"],
     "answer": ["제주시", "서귀포시"], "outcome_score": 0.85, "step_scores": [0.8, 0.8, 0.8],
     "process_review": "근거 범주와 명칭, 최종 답이 일치"},
    {"id": "C", "steps": ["공개 문서의 행정시 범주 확인", "제주시와 서귀포시 두 명칭 확인", "제주시와 성산읍으로 명칭을 잘못 옮겨 답변"],
     "answer": ["제주시", "성산읍"], "outcome_score": 0.2, "step_scores": [0.95, 0.95, 0.05],
     "process_review": "앞 2단계 타당, 3단계 명칭 전사 오류"},
    {"id": "D", "steps": ["공개 문서의 행정시 범주 확인", "제주시와 서귀포시 두 명칭 확인", "제주시와 제주시로 같은 명칭 두 번 답변"],
     "answer": ["제주시", "제주시"], "outcome_score": 0.6, "step_scores": [0.99, 0.99, 0.5],
     "process_review": "앞 2단계 타당, 3단계 중복 결론 오류"},
]
for row in candidates:
    row["answer_match"] = len(row["answer"]) == 2 and set(row["answer"]) == EXPECTED
    print(row["id"], "정답 대조:", row["answer_match"], "과정:", row["process_review"])
    for i, (step, score) in enumerate(zip(row["steps"], row["step_scores"]), 1):
        print(f"  단계 {i}: {step}, 교수 제공 점수={score}")
assert [row["answer_match"] for row in candidates] == [True, True, False, False]


A 정답 대조: True 과정: 2단계 범주 추론 오류. 최종 답 일치가 이 추론을 정당화하지 않음
  단계 1: 문서에서 제주시와 서귀포시 명칭 확인, 교수 제공 점수=0.9
  단계 2: 이름이 다른 지역은 모두 행정시라고 간주, 교수 제공 점수=0.1
  단계 3: 제주시와 서귀포시라고 답변, 교수 제공 점수=0.9
B 정답 대조: True 과정: 근거 범주와 명칭, 최종 답이 일치
  단계 1: 공개 문서의 행정시 범주 확인, 교수 제공 점수=0.8
  단계 2: 제주시와 서귀포시 두 명칭을 해당 범주와 연결, 교수 제공 점수=0.8
  단계 3: 제주시와 서귀포시라고 답변, 교수 제공 점수=0.8
C 정답 대조: False 과정: 앞 2단계 타당, 3단계 명칭 전사 오류
  단계 1: 공개 문서의 행정시 범주 확인, 교수 제공 점수=0.95
  단계 2: 제주시와 서귀포시 두 명칭 확인, 교수 제공 점수=0.95
  단계 3: 제주시와 성산읍으로 명칭을 잘못 옮겨 답변, 교수 제공 점수=0.05
D 정답 대조: False 과정: 앞 2단계 타당, 3단계 중복 결론 오류
  단계 1: 공개 문서의 행정시 범주 확인, 교수 제공 점수=0.99
  단계 2: 제주시와 서귀포시 두 명칭 확인, 교수 제공 점수=0.99
  단계 3: 제주시와 제주시로 같은 명칭 두 번 답변, 교수 제공 점수=0.5


## 3. 동일 후보에서 네 집계 비교

- 응답 점수: 제공된 `outcome_score` 최대 후보
- 평균: 단계 점수 합 / 단계 수
- 최솟값: 가장 낮은 단계 점수
- 곱: 세 단계 점수 곱. 점수 의존성·교정 없이 전체 정답 확률로 해석 금지
- 동점이면 후보 순서상 먼저 나온 후보 선택
- N은 제공 후보의 접두 집합 크기. 후보 생성 난수 실험 아님


In [3]:
def aggregate(row, method):
    values = row["step_scores"]
    if method == "outcome":
        return row["outcome_score"]
    if method == "mean":
        return mean(values)
    if method == "min":
        return min(values)
    if method == "product":
        return math.prod(values)
    raise ValueError("알 수 없는 집계")

comparisons = []
for n in (2, 3, 4):
    for method in ("outcome", "mean", "min", "product"):
        pool = candidates[:n]
        ranked = sorted(pool, key=lambda row: aggregate(row, method), reverse=True)
        winner = ranked[0]
        result = {"n": n, "method": method, "winner": winner["id"],
                  "score": aggregate(winner, method), "answer_match": winner["answer_match"],
                  "ranking": [[row["id"], aggregate(row, method)] for row in ranked]}
        comparisons.append(result)
        print("N =", n, method, "선택", winner["id"],
              "score =", round(result["score"], 6), "정답 대조 =", winner["answer_match"])
base_winners = {row["method"]: row["winner"] for row in comparisons if row["n"] == 4}
assert base_winners == {"outcome": "A", "mean": "D", "min": "B", "product": "B"}
assert math.isclose(aggregate(candidates[3], "mean"), 2.48 / 3)
assert math.isclose(aggregate(candidates[1], "product"), 0.512)
print("기본 N=4 검산:", base_winners)


N = 2 outcome 선택 A score = 0.95 정답 대조 = True
N = 2 mean 선택 B score = 0.8 정답 대조 = True
N = 2 min 선택 B score = 0.8 정답 대조 = True
N = 2 product 선택 B score = 0.512 정답 대조 = True
N = 3 outcome 선택 A score = 0.95 정답 대조 = True
N = 3 mean 선택 B score = 0.8 정답 대조 = True
N = 3 min 선택 B score = 0.8 정답 대조 = True
N = 3 product 선택 B score = 0.512 정답 대조 = True
N = 4 outcome 선택 A score = 0.95 정답 대조 = True
N = 4 mean 선택 D score = 0.826667 정답 대조 = False
N = 4 min 선택 B score = 0.8 정답 대조 = True
N = 4 product 선택 B score = 0.512 정답 대조 = True
기본 N=4 검산: {'outcome': 'A', 'mean': 'D', 'min': 'B', 'product': 'B'}


## 4. 집계가 신호 오판을 복구하는가?

- D의 최종 답·문장·정답 대조는 유지, 단계 점수만 모두 0.99로 변경
- 최솟값·곱도 D 선택. 좋은 집계 기준이 틀린 평가 신호를 항상 고치지는 못함
- 실제 PRM 오판율을 측정한 실험이 아니라 교수 작성 민감도 예제


In [4]:
unreliable = deepcopy(candidates)
unreliable[3]["step_scores"] = [0.99, 0.99, 0.99]
error_results = []
for method in ("mean", "min", "product"):
    selected = max(unreliable, key=lambda row: aggregate(row, method))
    error_results.append({"method": method, "winner": selected["id"],
                          "score": aggregate(selected, method), "answer_match": selected["answer_match"]})
    print("오판 preset:", method, selected["id"], round(aggregate(selected, method), 6), selected["answer_match"])
assert all(row["winner"] == "D" and not row["answer_match"] for row in error_results)
assert unreliable[3]["answer"] == candidates[3]["answer"]


오판 preset: mean D 0.99 False
오판 preset: min D 0.99 False
오판 preset: product D 0.970299 False


## 5. 곱의 길이 효과와 20단계 신용의 위치

- 각 단계 점수 0.8인 3단계와 6단계 비교, 평균·최솟값 동일, 곱은 감소
- 20단계 예제의 -1은 교수 제공 응답 상대 신호. 응답 실패=0에서 자동으로 도출되는 값 아님
- 그 신호가 모든 위치에 반복되어도 7번째 오류 위치를 자체적으로 알려주지는 않음
- 별도 교수 단계 라벨은 위치 7을 알려 주지만 토큰 인과 효과를 측정하지는 않음
- 실제 tokenizer·학습기·PRM 없음


In [5]:
length_effect = []
for length in (3, 6):
    scores = [0.8] * length
    item = {"length": length, "mean": mean(scores), "min": min(scores), "product": math.prod(scores)}
    length_effect.append(item)
    print("길이 집계:", item)
assert math.isclose(length_effect[0]["product"], 0.512)
assert math.isclose(length_effect[1]["product"], 0.262144)
response_signal = -1.0  # 교수 제공 상대 신호 예시, 실제 advantage 계산 아님
broadcast = [response_signal] * 20
teacher_step_labels = ["타당"] * 20
teacher_step_labels[6] = "범주 추론 오류"
for i, (signal, label) in enumerate(zip(broadcast, teacher_step_labels), 1):
    print(f"위치 {i:02d}: 응답 신호={signal:+.1f}, 별도 교수 단계 라벨={label}")
assert len(set(broadcast)) == 1
assert teacher_step_labels.index("범주 추론 오류") + 1 == 7


길이 집계: {'length': 3, 'mean': 0.8, 'min': 0.8, 'product': 0.5120000000000001}
길이 집계: {'length': 6, 'mean': 0.8, 'min': 0.8, 'product': 0.2621440000000001}
위치 01: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 02: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 03: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 04: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 05: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 06: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 07: 응답 신호=-1.0, 별도 교수 단계 라벨=범주 추론 오류
위치 08: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 09: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 10: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 11: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 12: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 13: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 14: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 15: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 16: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 17: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 18: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 19: 응답 신호=-1.0, 별도 교수 단계 라벨=타당
위치 20: 응답 신호=-1.0, 별도 교수 단계 라벨=타당


## 6. 관찰 결과 저장과 기록 양식

- 저장 자료: 출처 메타데이터, 교수 예시, 제공 점수, 산술 출력
- 공개 원문·수집 문단 전체는 JSON에 저장하지 않음
- 이름·연락처 미포함. 제출 폴더에는 학번만 사용

| 항목 | 학생 기록 |
|---|---|
| 동일 조건 | 고정판, 질문, 후보, 제공 점수 |
| 변경 조건 | N / 집계 / 오판 preset |
| 관찰 | 선택 후보, 점수, 정답 대조, 과정 검토 |
| 판정 | 점수 높음과 최종 답 일치·과정 타당성의 차이 |
| 한계 | 실제 PRM 출력·모델 생성·학습 효과는 미측정 |
| 신용 5요소 | 앞 네 설계 항목 검토, 학습 효과는 미측정 |

- 산출물: 아래 JSON과 위 표에 따른 짧은 관찰 기록
- 발표 문장 예: "제공값에서 평균은 D를, min/product는 B를 선택. 오판 신호에서는 둘 다 D를 선택하므로 실측 PRM 우위는 주장할 수 없음."


### 실행 후 작성한 관찰 기록

| 항목 | 실제 실행에 근거한 기록 |
|---|---|
| 동일 조건 | 위키백과 고정판 42464087의 제주시·서귀포시 근거, 같은 질문과 교수 작성 후보 A-D 사용. 기본 집계 비교에서는 각 후보의 제공 점수도 동일하게 유지 |
| 변경 조건 | N은 후보 접두 집합 2, 3, 4개. 집계는 outcome/mean/min/product. 별도 오판 조건에서는 D의 단계 점수만 [0.99, 0.99, 0.99]로 변경 |
| 관찰 | N=2와 3은 outcome A, 나머지 B. N=4는 outcome A(0.95), mean D(0.826667), min B(0.8), product B(0.512) 선택 |
| 판정 | A는 최종 두 도시 집합은 맞지만 2단계 범주 추론은 잘못된 교수 예시. D는 높은 평균 점수에도 최종 답에서 제주시를 중복해 오답. B는 제시된 과정 검토와 최종 답 모두 일치 |
| 한계 | 후보는 생성 표본이 아니고 점수는 실제 ORM/PRM 출력이나 교정된 확률이 아님. N 증가로 성능이 좋아진다는 일반화 불가. 모델 생성·정책 학습·학습 효과 미측정 |
| 신용 5요소 | 방향·크기·위치·신뢰성은 아래처럼 제공된 설계 조건을 검토. 학습 효과는 파라미터 업데이트가 없어 미측정 |

| 신용 요소 | 관찰 및 판단 |
|---|---|
| 방향 | 응답 신호 -1.0은 억제 방향의 상정 상대 신호. 최종 답 불일치에서 자동으로 계산한 advantage가 아님 |
| 크기 | 모든 20개 위치에 크기 1.0을 동일하게 배분한 예시. 추가 정규화·clipping이나 gradient 측정 없음 |
| 위치 | 응답 신호는 1-20 위치 모두 동일. 오류 위치 7은 별도 교수 단계 라벨에서만 알 수 있음 |
| 신뢰성 | 교수 제공 점수의 오판 조건에서 mean/min/product가 모두 틀린 D를 선택. 집계만으로 평가 신호의 신뢰성이 보장되지 않음 |
| 학습 효과 | 학습기·파라미터 업데이트가 없으므로 개선·악화 모두 미측정 |

- 실행 환경: Python 3.10.0, AMD64, 표준 라이브러리 CPU 관찰.
- 실제 원문 수집 시각: `2026-11-12T15:02:45.085287+00:00`.
- API 응답 SHA256: `8dd0d492441ff5c8b930eef682110b6736c472a119b0c935f3066462ac15f87f`. JSON에는 원문 전체나 수집 문단을 저장하지 않고 출처·라이선스·수집 조건을 보존.


In [6]:
report = {
    "kind": "교수 작성 후보와 상정 점수의 관찰, 실제 ORM/PRM 측정 아님",
    "execution_conditions": execution_conditions,
    "source_metadata": source_metadata,
    "public_fact": sorted(EXPECTED),
    "candidates": candidates,
    "comparisons": comparisons,
    "unreliable_scores": error_results,
    "length_effect": length_effect,
    "credit_example": {"response_signal": response_signal, "positions": 20, "teacher_error_position": 7},
    "learning_effect": "미측정: 파라미터 업데이트 없음",
}
path = Path("week-11-observations.json")
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
assert "text" not in source_metadata and "evidence" not in report
print("저장:", path, "공개 원문 전체 미포함, 학습 효과 미측정")


저장: week-11-observations.json 공개 원문 전체 미포함, 학습 효과 미측정


## 돌아보기: 확인 질문

1. 같은 후보에서 응답 점수·평균·최솟값·곱이 각각 선택하는 후보 확인
2. D의 점수만 바꾸었을 때 정답 대조와 선택이 다르게 움직이는 이유 설명
3. 0.8을 3번 곱한 값과 6번 곱한 값에서 길이 효과 확인
4. 20개 위치에 같은 응답 신호가 있어도 7번째 오류 위치를 알아내지 못하는 이유 설명

## 제출 안내

- 코드 구현이나 자료 수집을 새로 요구하는 과제가 아닌 제공된 실습의 관찰 기록
- 실행 결과 JSON과 질문별 관찰·판정 이유를 함께 제출
- 저장된 결과에서 미확인·미판정은 그대로 표시, 모델 학습 성과로 보고하지 않는 원칙
- 학생 폴더에는 학번만 사용, 이름·연락처·데이터 원본·API 키 제외


### 확인 질문 답

1. **같은 후보에서 집계별 선택.** N=2와 N=3에서는 응답 점수가 A, 평균·최솟값·곱이 B를 선택. N=4에서는 아래 결과가 관찰됨. A의 정답 대조 True가 과정의 타당성을 보장하지 않고, D의 평균 우위가 최종 정답을 보장하지 않음.

| N=4 집계 | 선택 후보 | 점수 | 최종 답 대조 | 교수 과정 검토 |
|---|---|---:|---|---|
| 응답 점수 | A | 0.950000 | True | 2단계 범주 추론 오류 |
| 단계 평균 | D | 0.826667 | False | 마지막에 제주시 중복 |
| 단계 최솟값 | B | 0.800000 | True | 근거 범주와 두 명칭 연결 |
| 단계 곱 | B | 0.512000 | True | 근거 범주와 두 명칭 연결 |

2. **D의 점수만 바뀐 오판 조건.** D의 문장과 답 [제주시, 제주시]는 그대로여서 정답 대조는 False 유지. 단계 점수만 [0.99, 0.99, 0.99]가 되면 평균 0.99, 최솟값 0.99, 곱 0.970299로 세 집계 모두 D를 선택. 선택 함수는 제공 점수를 보고 순위를 매기지만 정답 대조는 공개 근거의 두 도시 집합을 확인하므로 서로 다르게 움직임. 이 예시는 집계가 잘못된 평가 신호를 항상 고치지 못함을 보여 주며 실제 PRM 오판율의 측정은 아님.

3. **곱의 길이 효과.** 0.8을 3번 곱한 값은 0.512, 6번 곱한 값은 0.262144. 단계 수를 2배로 늘렸을 때 곱은 이전의 0.512배로 감소하고 평균과 최솟값은 모두 0.8 유지. 0과 1 사이 점수의 곱은 단계 수에 민감하므로 서로 다른 길이의 후보를 비교할 때 불리한 길이 효과가 발생할 수 있음. 단계 간 독립성·확률 교정을 확인하지 않았으므로 이 곱을 전체 정답 확률로 해석할 수 없음.

4. **동일한 응답 신호와 오류 위치의 차이.** 20개 위치 모두 -1.0이므로 이 신호 안에는 7번째 위치를 다른 위치와 구별할 정보가 없음. 위치 7의 범주 추론 오류는 별도 교수 단계 라벨이 제공한 정보이며, 모든 위치에 같은 신호가 있다는 사실에서 얻은 결론이 아님. 그 라벨도 해당 위치를 바꾸었을 때 결과가 어떻게 달라지는지 검증한 토큰 수준 인과 효과는 아니고, 이번 실습에는 tokenizer·학습기·실제 PRM 실행이 없음.
